In [0]:
import mlflow
import os
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline

In [0]:
email_data = spark.read.parquet(
    "/Volumes/workspace/default/email_warehouse/silver/clean_emails"
)

In [0]:
with mlflow.start_run():
    mlflow.log_param("test_param", 123)
    mlflow.log_metric("test_metric", 0.95)

In [0]:
from pyspark.sql.functions import lower, col, size, split

# Convert body text to lowercase
emails = email_data.withColumn(
    "body_lower",
    lower(col("body"))
)

# Define sensitive keywords
sensitive_keywords = [
    "password",
    "confidential",
    "ssn",
    "social security",
    "credit card",
    "account number",
    "wire transfer",
    "bank details",
    "private",
    "urgent transfer",
    "otp",
    "signature",
    "pin",
    "bank statement",
    "card number",
    "cvv",
    "bank pin",
    "banking",
    "bank account",
    "banking details",
    "banking information",
    "banking credentials",
    "banking login",
    "banking password",
    "banking username"
]

# Create flag column
from pyspark.sql.functions import expr

keyword_pattern = "|".join(sensitive_keywords)

emails = emails.withColumn(
    "sensitive_flag",
    col("body_lower").rlike(keyword_pattern)
)

emails.select("sensitive_flag").groupBy("sensitive_flag").count().display()

In [0]:
# Count number of sensitive keywords
emails = emails.withColumn(
    "risk_score",
    size(split(col("body_lower"), keyword_pattern)) - 1
)

emails.select("risk_score").describe().display()

In [0]:
emails.display(5)

In [0]:
# SSN pattern (XXX-XX-XXXX)
ssn_pattern = r"\b\d{3}-\d{2}-\d{4}\b"

# Credit card pattern (16 consecutive digits)
credit_card_pattern = r"\b\d{16}\b"

emails = emails.withColumn(
    "ssn_detected",
    col("body_lower").rlike(ssn_pattern)
)

emails = emails.withColumn(
    "creditcard_detected",
    col("body_lower").rlike(credit_card_pattern)
)

emails.groupBy("ssn_detected").count().display()
emails.groupBy("creditcard_detected").count().display()

In [0]:
# calculate final risk score
emails = emails.withColumn(
    "final_risk_score",
    col("risk_score") +
    col("ssn_detected").cast("int") * 5 +
    col("creditcard_detected").cast("int") * 5
)

emails.select("final_risk_score").describe().display()
emails.groupBy("final_risk_score").count().display()

In [0]:
from pyspark.sql.functions import when

# calculate final risk level
emails = emails.withColumn(
    "risk_level",
    when(col("final_risk_score") == 0, "Non-Sensitive")
    .otherwise("Sensitive")
)

emails.groupBy("risk_level").count().display()

In [0]:
#encode risk level
emails = emails.withColumn(
    "label",
    when(col("risk_level") == "Non-Sensitive", 0)
    .otherwise(1)
)


emails.select("risk_level", "label").display()

In [0]:
#Tokenize the text(break email into words)
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="body_lower",
    outputCol="words"
)

emails = tokenizer.transform(emails)

In [0]:
emails.display()

In [0]:
#Remove stopwords
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

emails = remover.transform(emails)

In [0]:
emails.display()

In [0]:
#Convert words to numerical frequency
from pyspark.ml.feature import CountVectorizer

vectorizer = CountVectorizer(
    inputCol = "filtered_words",
    outputCol = "raw_features",
    vocabSize = 5000,
    minDF=3
)

cv_model = vectorizer.fit(emails)
emails = cv_model.transform(emails)

In [0]:
emails.display()

In [0]:
#Apply TF-IDF
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

idf_model = idf.fit(emails)
emails = idf_model.transform(emails)


In [0]:
#Train test split
train_data, test_data = emails.randomSplit([0.8, 0.2], seed=42)

print('Training rows:', train_data.count())
print('Testing rows:', test_data.count())

In [0]:
test_data.display()

In [0]:
#Train logistic regression
from pyspark.ml.classification import LogisticRegression

log = LogisticRegression(
    featuresCol = 'features',
    labelCol = 'label',
    maxIter = 20,
)

log_model = log.fit(train_data) 

In [0]:
prediction = log_model.transform(test_data)
prediction.display()

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(prediction)

print("Model Accuracy:", accuracy)

In [0]:
prediction.groupby('label', 'prediction').count().orderBy('label').display()

In [0]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

#Extract sensitive probability

prob = udf(lambda v: float(v[1]), DoubleType())

prediction = prediction.withColumn(
    "prob_sensitive",
    prob(col("probability"))
)

In [0]:
prediction.display()

In [0]:
#Define sensitivity probability
prediction = prediction.withColumn(
    "ml_risk_level",
    when(col("prob_sensitive") >= 0.85, "High")
    .when(col("prob_sensitive") >= 0.50, "Medium")
    .otherwise("Low")
)

In [0]:
#Add rule override
prediction = prediction.withColumn(
    "final_prediction",
    when(col("ssn_detected") == True, 1)
    .when(col("creditcard_detected") == True, 1)
    .otherwise(col("prediction"))
)

In [0]:
len(cv_model.vocabulary)

In [0]:
coefficients = log_model.coefficients.toArray()
len(coefficients)

In [0]:
word_importance = list(zip(cv_model.vocabulary, coefficients))

In [0]:
top_sensitive_words = sorted(word_importance, key=lambda x: x[1], reverse=True)[:15]
top_sensitive_words

In [0]:
top_non_sensitive_words = sorted(word_importance, key=lambda x: x[1], reverse=False)[:20]
top_non_sensitive_words

In [0]:
import pandas as pd

In [0]:
top_sensitive_df = pd.DataFrame(top_sensitive_words, columns=['word', 'importance'])

In [0]:
top_sensitive_df.display()

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.barh(top_sensitive_df["word"], top_sensitive_df["importance"])

plt.xlabel("Coefficient Weight")
plt.ylabel("Word")
plt.title("Top 20 Sensitive-Indicating Words")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

In [0]:
#Production demo

def predict_single_email(email_text):
    #Create Spark DataFrame with one row (column must be 'body' to match training pipeline)
    data = spark.createDataFrame([Row(body_lower=email_text.lower())])
    
    #Apply same preprocessing pipeline
    data = tokenizer.transform(data)
    data = remover.transform(data)
    data = cv_model.transform(data)
    data = idf_model.transform(data)
    
    #Run model prediction
    prediction = log_model.transform(data)
    
    #Extract probability of Sensitive class
    extract_prob = udf(lambda v: float(v[1]), DoubleType())
    
    prediction = prediction.withColumn(
        "prob_sensitive",
        extract_prob(col("probability"))
    )
    
    #Create ML risk level
    prediction = prediction.withColumn(
        "ml_risk_level",
        when(col("prob_sensitive") >= 0.85, "High")
        .when(col("prob_sensitive") >= 0.50, "Medium")
        .otherwise("Low")
    )
    
    #Rule-based override (regex detection example)
    import re
    
    ssn_pattern = r"\b\d{3}-\d{2}-\d{4}\b"
    credit_pattern = r"\b\d{4}-\d{4}-\d{4}-\d{4}\b"
    
    has_ssn = 1 if re.search(ssn_pattern, email_text) else 0
    has_credit = 1 if re.search(credit_pattern, email_text) else 0
    
    final_prediction = 1 if (has_ssn or has_credit) else prediction.select("prediction").collect()[0][0]
    
    prob = prediction.select("prob_sensitive").collect()[0][0]
    ml_risk = prediction.select("ml_risk_level").collect()[0][0]
    
    return {
        "prediction": "Sensitive" if final_prediction == 1 else "Non-Sensitive",
        "probability_sensitive": round(prob, 4),
        "ml_risk_level": ml_risk,
        "rule_override_applied": True if (has_ssn or has_credit) else False
    }


In [0]:
predict_single_email('Please send your password and bank details immediately')

In [0]:
#Final model evaluation
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

roc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

print("Accuracy:", accuracy_eval.evaluate(prediction))
print("Precision:", precision_eval.evaluate(prediction))
print("Recall:", recall_eval.evaluate(prediction))
print("F1 Score:", f1_eval.evaluate(prediction))
print("ROC AUC:", roc_eval.evaluate(prediction))

In [0]:
prediction.groupBy("label", "prediction").count().orderBy("label", "prediction").display()

In [0]:
# Save models in silver folder, same as clean_data
#silver_path = "/Volumes/workspace/default/email_warehouse/silver/models/"
#tokenizer.write().overwrite().save(silver_path + "tokenizer")
#remover.write().overwrite().save(silver_path + "remover")
#cv_model.write().overwrite().save(silver_path + "cv_model")
#idf_model.write().overwrite().save(silver_path + "idf_model")
#log_model.write().overwrite().save(silver_path + "log_model")

In [0]:
#Define input and output format (signature)
input_example = spark.createDataFrame(
    [("please send your password",)],
    ["body_lower"]
)

# run through preprocessing pipeline
tok = tokenizer.transform(input_example)
sw = remover.transform(tok)
cv = cv_model.transform(sw)
idf = idf_model.transform(cv)

# prediction
example_output = log_model.transform(idf)

In [0]:
pipeline = Pipeline(stages=[tokenizer, remover, cv_model, idf_model, log_model])

In [0]:
pipeline_model = pipeline.fit(train_data)

In [0]:
signature = infer_signature(input_example.toPandas(), example_output.select('prediction').toPandas())

In [0]:
#Adopting mlflow

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/email_warehouse/tmp"

with mlflow.start_run():

    mlflow.log_param("model_type", "logistic_regression")

    mlflow.log_metric("accuracy", accuracy)

    mlflow.spark.log_model(spark_model = pipeline_model, 
                           artifact_path = "email_sensitivity_classifier",
                           signature = signature,
                           input_example = input_example.toPandas(),
                           pip_requirements = ['pyspark==3.5.0']
                           )

In [0]:
dbutils.secrets.listScopes()